# Etapa 4 (Parte 2) — Variável oculta Z e algoritmo EM

**Como usar este notebook:** rode as células de cima para baixo (clique na célula e aperte `Shift + Enter`, ou use *Run All*). Leva menos de um minuto.

**Preparação:** no GitHub, os arquivos `em.py` e `trabalho_etapas_3_4_bayes.py` ficam na raiz e este notebook em `notebooks/`; o notebook sobe sozinho para a pasta principal. A tabela `dados/tabela_p4.csv` (feita na Etapa 1) é procurada na pasta atual e em `dados/`; se não for encontrada, o módulo da Membro 3 baixa do GitHub do grupo. No Google Colab, envie `em.py` e `trabalho_etapas_3_4_bayes.py` para `/content` (e, se quiser, a `tabela_p4.csv`), mesmo que abra o notebook direto pelo GitHub.

**Bibliotecas:** numpy, pandas, matplotlib e scikit-learn.

**O que ele cria:** `loglik_em.png` (log-verossimilhança do EM) e `comparacao_nb_em.png` (Naive Bayes puro vs. com Z), na pasta principal.

**A ideia:** existe uma "condição climática" que ninguém anotou (um grupo escondido, **Z**). Só temos três números de clima por auto de infração: chuva, temperatura e umidade. O algoritmo **EM** descobre, só olhando esses números, a que grupo cada auto pertence. Depois testamos se saber o grupo ajuda o Naive Bayes da Membro 3 a prever `Multa_Alta`.

**Roteiro:**
1. Carregar os dados (os mesmos da Membro 3).
2. Testar o EM em dados inventados.
3. Rodar o EM nos dados reais e interpretar os grupos.
4. Comparar Naive Bayes puro vs. com Z (validação cruzada de 5 folds).
5. Conferir no teste externo (30% separado).

In [ ]:
import os, sys
# Se este notebook estiver em notebooks/, sobe uma pasta (onde ficam em.py e o módulo da Membro 3)
if not os.path.exists("em.py") and os.path.exists("../em.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta de trabalho:", os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

import trabalho_etapas_3_4_bayes as projeto          # Naive Bayes do zero (Membro 3)
from em import MisturaGaussianaEM, adicionar_Z, z_para_matriz, plotar_loglik, teste_sintetico

SEED = 42
np.random.seed(SEED)

## Passo 1 — Os dados

Usamos exatamente o pipeline da **Membro 3** (a mesma `tabela_p4.csv` da Etapa 1), para que a comparação seja justa:
- 70% treino / 30% teste externo, estratificado pela classe, semente 42;
- alvo `Multa_Alta` = valor da multa acima da mediana (limiar calculado com todos os autos da tabela);
- o Naive Bayes dela usa 6 atributos categóricos, incluindo as faixas de clima (`CHUVA_FAIXA`, `TEMP_FAIXA`, `UMID_FAIXA`).

A entrada do EM são os **números** de clima (`precip_mm`, `temp_media`, `umid_media`), que já estão na tabela. Autos sem clima conhecido (cerca de 7%) terão Z = `"sem_dado"`.

In [ ]:
COLS_CLIMA = ["precip_mm", "temp_media", "umid_media"]            # entrada do EM

df_treino_bruto = projeto.df_treino_bruto.reset_index(drop=True)
df_teste_bruto = projeto.df_teste_bruto.reset_index(drop=True)
clima_treino = df_treino_bruto[COLS_CLIMA]
clima_teste = df_teste_bruto[COLS_CLIMA]

print("Autos no treino:", len(df_treino_bruto), "| no teste externo:", len(df_teste_bruto))
print("Treino com clima completo:", f"{clima_treino.notna().all(axis=1).mean():.1%}")
print("Limiar de multa alta:", projeto.formatar_reais(projeto.LIMIAR_MULTA))
clima_treino.describe().round(1)

## Passo 2 — Testando o EM em dados inventados

Antes de confiar no EM com dados reais, geramos 2 grupos **cujas respostas já sabemos** (médias [0,0] e [5,4], pesos 60% e 40%) e vemos se o algoritmo as encontra.

O que conferir na saída:
- as médias e os pesos achados ficam perto dos reais;
- a log-verossimilhança **só sobe** (se cair em algum ponto, há bug).

In [ ]:
m_teste = teste_sintetico()
plotar_loglik(m_teste.loglik_, titulo="Teste sintético: log-verossimilhança")

## Passo 3 — EM nos dados reais

Padronizamos os 3 números de clima (média 0, desvio 1) para que nenhum domine só por ter escala maior. Rodamos o EM com **K = 2** grupos nos autos de **treino** e voltamos as médias para as unidades originais para conseguir **dar nome** aos grupos.

In [ ]:
d = clima_treino.dropna()
media, desvio = d.values.mean(axis=0), d.values.std(axis=0)
K = 2
em = MisturaGaussianaEM(K=K, seed=SEED).fit((d.values - media) / desvio)

print(f"Convergiu em {em.n_iter_} iterações")
print("A log-verossimilhança só sobe?", bool(np.all(np.diff(em.loglik_) > -1e-8)))
tabela = pd.DataFrame(em.mu_ * desvio + media, columns=COLS_CLIMA, index=[f"Z{k}" for k in range(K)]).round(1)
tabela.insert(0, "peso", np.round(em.theta_, 3))
tabela

In [ ]:
plotar_loglik(em.loglik_, titulo=f"EM (K={K}) nos dados reais: log-verossimilhança", salvar="loglik_em.png")

**Como ler o resultado:** olhe a tabela e dê um nome para cada grupo (por exemplo, "regime chuvoso e úmido" e "regime mais seco"). Esse nome é a interpretação da variável oculta e vai para o relatório.

O gráfico deve crescer rápido no começo e depois ficar quase reto. O gabarito espera estabilizar entre 15 e 30 iterações.

## Passo 4 — Naive Bayes puro vs. com Z (validação cruzada de 5 folds)

Usamos o `NaiveBayesRaiz` da Membro 3. Ele trabalha com atributos 0/1, então o Z entra como colunas 0/1 (`Z0`, `Z1`, ... e `sem_dado`), coladas nas colunas que ela já tem.

Comparamos quatro configurações:

| Configuração | O que muda |
|---|---|
| **Puro** | só os atributos da Membro 3 |
| **+ Z (K=2)** | os atributos dela mais o Z com 2 grupos |
| **+ Z (K=3)** | os atributos dela mais o Z com 3 grupos |
| **Z no lugar das faixas de clima (K=2)** | tira `CHUVA_FAIXA`, `TEMP_FAIXA` e `UMID_FAIXA` e usa só o Z como resumo do clima |

Os folds são os mesmos dela (`StratifiedKFold`, 5 partes, semente 42). A configuração **Puro** deve reproduzir a acurácia da validação cruzada dela.

⚠️ **Sem vazamento:** dentro de cada fold, o pré-processamento dela e o EM são ajustados **só nos dados de treino do fold** e depois aplicados à validação.

In [ ]:
FAIXAS_CLIMA = ("CHUVA_FAIXA", "TEMP_FAIXA", "UMID_FAIXA")

def cv_acuracia(K, trocar_faixas=False, k=5):
    """Acurácia por fold. K=0: Naive Bayes puro. K>=2: Naive Bayes + Z com K grupos."""
    X_raw = df_treino_bruto[projeto.ATRIBUTOS_CATEGORICOS]
    y_cont = df_treino_bruto["VALOR_NUMERICO"]
    y_classe = (y_cont > projeto.LIMIAR_ETAPA1_CONFIRMADO).astype(int)
    acuracias = []
    for i_fit, i_val in StratifiedKFold(n_splits=k, shuffle=True, random_state=projeto.SEED).split(X_raw, y_classe):
        Xf, Xv, yf, yv, colunas, _, _ = projeto.preparar_particoes_ml(
            X_raw.iloc[i_fit], X_raw.iloc[i_val], y_cont.iloc[i_fit], y_cont.iloc[i_val])
        if K:
            if trocar_faixas:                                       # tira as colunas das faixas de clima
                manter = ~pd.Index(colunas).str.startswith(FAIXAS_CLIMA)
                Xf, Xv = Xf[:, manter], Xv[:, manter]
            cf, cv_, _ = adicionar_Z(clima_treino.iloc[i_fit].reset_index(drop=True),
                                     clima_treino.iloc[i_val].reset_index(drop=True),
                                     COLS_CLIMA, K=K)               # EM só no treino do fold
            Xf = np.hstack([Xf, z_para_matriz(cf["Z"], K)])
            Xv = np.hstack([Xv, z_para_matriz(cv_["Z"], K)])
        modelo = projeto.NaiveBayesRaiz().fit(Xf, yf)
        acuracias.append(projeto.acuracia(yv, modelo.predict(Xv)))
    return np.array(acuracias)

configs = [("Naive Bayes puro", 0, False), ("+ Z (K=2)", 2, False), ("+ Z (K=3)", 3, False),
           ("Z no lugar das faixas (K=2)", 2, True)]
linhas = []
for nome, K, troca in configs:
    a = cv_acuracia(K, troca)
    linhas.append({"Configuração": nome, "Acurácia média": a.mean(), "Desvio": a.std(ddof=1)})
    print(f"{nome:28s} {a.mean():.2%} ± {a.std(ddof=1):.2%}")

resultado_cv = pd.DataFrame(linhas).round(4)

# Conferência: o "puro" daqui deve ser igual ao da Membro 3
acs_membro3, _ = projeto.validacao_cruzada(projeto.df_treino_bruto, k=5)
print("\nPuro bate com a validação cruzada da Membro 3?", bool(np.isclose(np.mean(acs_membro3), linhas[0]["Acurácia média"])))
resultado_cv

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(resultado_cv["Configuração"], resultado_cv["Acurácia média"], yerr=resultado_cv["Desvio"], capsize=4)
plt.ylabel("Acurácia média (5-fold)")
plt.title("Naive Bayes: puro vs. com Z")
plt.xticks(rotation=15, ha="right")
lo = max(0, resultado_cv["Acurácia média"].min() - 0.05)
plt.ylim(lo, min(1, resultado_cv["Acurácia média"].max() + 0.05))
plt.tight_layout()
plt.savefig("comparacao_nb_em.png", dpi=150)
plt.show()

## Passo 5 — Conferência no teste externo (30%)

Os 30% de autos que ficaram de fora de tudo. Aqui o EM e o Naive Bayes são ajustados em **todo o treino** e avaliados uma única vez no teste. Serve como conferência final do resultado da validação cruzada.

In [ ]:
linhas = []
for nome, K, troca in configs:
    Xf, Xt = projeto.X_tr, projeto.X_te
    if K:
        if troca:
            manter = ~pd.Index(projeto.ATRIBUTOS).str.startswith(FAIXAS_CLIMA)
            Xf, Xt = Xf[:, manter], Xt[:, manter]
        ctr, cte, _ = adicionar_Z(clima_treino, clima_teste, COLS_CLIMA, K=K)
        Xf = np.hstack([Xf, z_para_matriz(ctr["Z"], K)])
        Xt = np.hstack([Xt, z_para_matriz(cte["Z"], K)])
    modelo = projeto.NaiveBayesRaiz().fit(Xf, projeto.y_tr)
    linhas.append({"Configuração": nome, "Acurácia no teste": projeto.acuracia(projeto.y_te, modelo.predict(Xt))})

pd.DataFrame(linhas).round(4)

## Passo 6 — O que escrever no relatório

Escreva 2 a 3 parágrafos com as suas próprias palavras. Perguntas que ajudam:

1. **O que foi feito?** Qual variável oculta Z foi definida, com quantos grupos e a partir de quais colunas?
2. **Os grupos fazem sentido?** Que nome você deu a cada um, olhando a tabela do Passo 3?
3. **O EM funcionou?** Em quantas iterações convergiu? A log-verossimilhança subiu sem cair?
4. **Z ajudou o Naive Bayes?** Compare as acurácias dos Passos 4 e 5. O gabarito do enunciado cita 72–78% para o Naive Bayes puro e 80–86% com EM, mas são **faixas de referência, não obrigações**.
5. **Por que o Z ajudou pouco (ou nada)?** Pistas: o clima do INMET é um valor por estado e mês, e o Naive Bayes já recebe as faixas de clima, então o Z traz pouca informação nova. Além disso, o clima de um mês inteiro dificilmente explica o valor de uma multa individual.
6. **O que aprendeu?** Por exemplo: o EM é sensível à inicialização, variância perto de zero quebra o código, e ajustar o EM fora do treino gera vazamento.

Não é um problema o Z não melhorar a acurácia: um resultado negativo bem explicado vale mais do que um ganho que não se sustenta.

## Apêndice — Prompts usados

Se você usou um chatbot, o enunciado exige documentar os prompts. Cole aqui os principais:

- Prompt 1: …
- Prompt 2: …